# PutStrike iTransformer — Universal Multi-Stock Training

**iTransformer: Inverted Transformers Are Effective for Time Series Forecasting** (ICLR 2024, Liu et al.)

## Overview

Train a **single universal model** on all 80+ screener stocks simultaneously, then export to ONNX and push to HuggingFace Hub for serverless inference on the PutStrike website.

### Architecture
- **iTransformer**: each **feature** is a token (cross-variate attention captures how features interact)
- **RevIN**: Reversible Instance Normalization for non-stationary financial time series
- **100+ features**: OHLCV technicals + macro indicators (VIX, Treasury yields, USD, Gold, Oil)
- **60-day lookback → 60 trading day forecast** (covers all DTE presets: 7d to 120d)

### Training Design
- **Universal model**: trained on all stocks simultaneously (~200K samples vs ~2,500 per stock)
- **Walk-forward validation**: 70/15/15 chronological split (no look-ahead bias)
- **HuberLoss(delta=0.02)**: robust to earnings/event return outliers
- **LR warmup + cosine decay**: standard for Transformers
- **10 years of daily data** per stock via Yahoo Finance

### Output
- ONNX model (~2-5 MB) pushed to HuggingFace Hub
- Config JSON with feature names, normalization stats, architecture details
- No Flask/ngrok needed — website loads ONNX directly from HuggingFace

### Setup
1. Run in Google Colab with **L4 GPU** runtime (Runtime > Change runtime type > L4 GPU)
2. Set `HF_TOKEN` and `HF_REPO_ID` in Cell 2
3. Execute all cells in order (~30-60 min on L4 GPU)
4. Model auto-exports to ONNX and pushes to HuggingFace Hub

In [ ]:
# Cell 1: Install Dependencies
import subprocess
import sys

packages = [
    "torch", "numpy", "pandas", "yfinance",
    "scikit-learn", "matplotlib", "onnx", "onnxruntime",
    "onnxscript", "huggingface_hub",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("[OK] All dependencies installed.")

In [ ]:
# Cell 2: Configuration
import os
import torch
import numpy as np

# ── HuggingFace Configuration ──
# Set your HuggingFace token: https://huggingface.co/settings/tokens
HF_TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN_HERE")
HF_REPO_ID = os.environ.get("HF_REPO_ID", "jcl347/putstrike")

# ── All 80+ screener stocks (universal model) ──
SCREENER_SYMBOLS = [
    # Mega-cap tech
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "AVGO", "ORCL", "CRM",
    "AMD", "INTC", "QCOM", "ADBE", "NFLX", "CSCO", "IBM", "TXN", "NOW", "AMAT",
    "MU", "LRCX", "KLAC", "SNPS", "CDNS", "PANW", "CRWD", "FTNT",
    # Finance
    "JPM", "V", "MA", "BAC", "WFC", "GS", "MS", "AXP", "BLK", "SCHW", "C",
    # Healthcare
    "JNJ", "UNH", "LLY", "PFE", "ABBV", "MRK", "TMO", "ABT", "DHR", "BMY", "AMGN",
    # Consumer
    "PG", "KO", "PEP", "COST", "WMT", "MCD", "NKE", "SBUX", "TGT", "HD", "LOW",
    # Energy
    "XOM", "CVX", "COP", "SLB", "EOG",
    # Industrial
    "CAT", "DE", "HON", "UNP", "RTX", "BA", "GE", "LMT", "MMM",
    # ETFs
    "SPY", "QQQ", "IWM", "DIA", "XLF", "XLE", "XLK", "XLV", "XBI",
    # Other
    "DIS", "PYPL", "SQ", "COIN", "ABNB", "UBER",
]

# ── Macro tickers (additional features) ──
MACRO_TICKERS = ["^VIX", "^VIX3M", "^TNX", "DX-Y.NYB", "GC=F", "CL=F"]

# ── Architecture hyperparameters ──
LOOKBACK_WINDOW = 60       # 60 trading days (~3 months) lookback
FORECAST_HORIZON = 60      # 60 trading days forecast (covers all DTE presets up to 120d calendar)
D_MODEL = 128              # Transformer hidden dimension
N_HEADS = 8                # Attention heads
N_LAYERS = 3               # Transformer encoder layers
D_FF = 256                 # Feed-forward dimension
DROPOUT = 0.2              # Higher dropout for noisy financial data

# ── Training hyperparameters ──
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 3e-4       # Standard for Transformers
WARMUP_EPOCHS = 5          # LR warmup
WEIGHT_DECAY = 1e-3        # Regularization
PATIENCE = 15              # Early stopping patience
TRAIN_SPLIT = 0.70         # 70% train
VAL_SPLIT = 0.85           # Of remaining 30%, 15% val + 15% test
DATA_YEARS = 10            # Years of historical data per stock

# ── Reproducibility ──
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[CONFIG] {len(SCREENER_SYMBOLS)} stocks, {len(MACRO_TICKERS)} macro indicators")
print(f"[CONFIG] lookback={LOOKBACK_WINDOW}, horizon={FORECAST_HORIZON}")
print(f"[CONFIG] d_model={D_MODEL}, layers={N_LAYERS}, heads={N_HEADS}, dropout={DROPOUT}")
print(f"[DEVICE] {device}" + (f" — {torch.cuda.get_device_name(0)}" if device.type == 'cuda' else ""))

In [ ]:
# Cell 3: Feature Engineering
# Computes 100+ features from OHLCV data + macro indicators

import pandas as pd
import yfinance as yf
from typing import Dict, List, Optional, Tuple

def compute_features(df: pd.DataFrame, macro_df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """
    Compute 100+ features from OHLCV data plus macro indicators.
    Mirrors the TypeScript feature engineering in src/lib/features.ts.
    """
    feat = pd.DataFrame(index=df.index)
    close = df["Close"].squeeze()
    high = df["High"].squeeze()
    low = df["Low"].squeeze()
    volume = df["Volume"].squeeze()
    open_ = df["Open"].squeeze()

    # ── Moving Averages (10 features) ──
    for p in [5, 10, 20, 50, 200]:
        sma = close.rolling(p).mean()
        feat[f"price_vs_sma_{p}_pct"] = ((close - sma) / sma) * 100

    for p in [5, 12, 26]:
        ema_val = close.ewm(span=p, adjust=False).mean()
        feat[f"price_vs_ema_{p}_pct"] = ((close - ema_val) / ema_val) * 100

    feat["sma_20_50_cross"] = (close.rolling(20).mean() > close.rolling(50).mean()).astype(float)
    feat["sma_50_200_cross"] = (close.rolling(50).mean() > close.rolling(200).mean()).astype(float)

    # ── RSI (3 features) ──
    for p in [7, 14, 21]:
        delta = close.diff()
        gain = delta.where(delta > 0, 0).rolling(p).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(p).mean()
        rs = gain / (loss + 1e-10)
        feat[f"rsi_{p}"] = 100 - 100 / (1 + rs)

    # ── MACD (2 features) ──
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    macd_signal = macd_line.ewm(span=9, adjust=False).mean()
    feat["macd_histogram"] = (macd_line - macd_signal) / close * 100
    feat["macd_cross_above"] = ((macd_line > macd_signal) & (macd_line.shift(1) <= macd_signal.shift(1))).astype(float)

    # ── Bollinger Bands (2 features) ──
    bb_sma = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    feat["bb_width"] = (4 * bb_std / bb_sma) * 100
    feat["bb_pctb"] = (close - (bb_sma - 2 * bb_std)) / (4 * bb_std + 1e-10)

    # ── ATR (4 features) ──
    for p in [7, 14]:
        tr1 = high - low
        tr2 = (high - close.shift(1)).abs()
        tr3 = (low - close.shift(1)).abs()
        tr = np.maximum(np.maximum(tr1, tr2), tr3)
        atr = tr.rolling(p).mean()
        feat[f"atr_{p}_pct"] = (atr / close) * 100

    # ── Volume (5 features) ──
    feat["volume_ratio_5_20"] = volume.rolling(5).mean() / (volume.rolling(20).mean() + 1)
    feat["relative_volume"] = volume / (volume.rolling(20).mean() + 1)
    obv = (np.sign(close.diff()) * volume).cumsum()
    obv_norm = (obv - obv.rolling(20).mean()) / (obv.rolling(20).std() + 1e-10)
    feat["obv_zscore"] = obv_norm
    clv = ((close - low) - (high - close)) / (high - low + 1e-10)
    feat["cmf_20"] = (clv * volume).rolling(20).sum() / (volume.rolling(20).sum() + 1)
    feat["volume_zscore_20"] = (volume - volume.rolling(20).mean()) / (volume.rolling(20).std() + 1e-10)

    # ── Stochastic (2 features) ──
    low14 = low.rolling(14).min()
    high14 = high.rolling(14).max()
    feat["stoch_k"] = ((close - low14) / (high14 - low14 + 1e-10)) * 100
    feat["stoch_d"] = feat["stoch_k"].rolling(3).mean()

    # ── Williams %R ──
    feat["williams_r"] = ((high14 - close) / (high14 - low14 + 1e-10)) * -100

    # ── ROC (3 features) ──
    for p in [5, 10, 20]:
        feat[f"roc_{p}"] = close.pct_change(p) * 100

    # ── CCI ──
    tp = (high + low + close) / 3
    tp_sma = tp.rolling(20).mean()
    tp_mad = tp.rolling(20).apply(lambda x: np.mean(np.abs(x - np.mean(x))))
    feat["cci_20"] = (tp - tp_sma) / (0.015 * tp_mad + 1e-10)

    # ── Aroon (3 features) ──
    feat["aroon_up"] = high.rolling(25).apply(lambda x: x.argmax() / 24 * 100)
    feat["aroon_down"] = low.rolling(25).apply(lambda x: x.argmin() / 24 * 100)
    feat["aroon_oscillator"] = feat["aroon_up"] - feat["aroon_down"]

    # ── Returns (5 features) ──
    for p in [1, 5, 10, 20, 60]:
        feat[f"return_{p}d"] = close.pct_change(p)

    # ── Volatility (4 features) ──
    log_ret = np.log(close / close.shift(1))
    for p in [5, 10, 20, 60]:
        feat[f"volatility_{p}d"] = log_ret.rolling(p).std() * np.sqrt(252)

    # ── Higher Moments (4 features) ──
    feat["skewness_20d"] = log_ret.rolling(20).skew()
    feat["skewness_60d"] = log_ret.rolling(60).skew()
    feat["kurtosis_20d"] = log_ret.rolling(20).kurt()
    feat["kurtosis_60d"] = log_ret.rolling(60).kurt()

    # ── Autocorrelation (3 features) ──
    for lag in [1, 3, 5]:
        feat[f"autocorr_lag_{lag}"] = log_ret.rolling(30).apply(
            lambda x: x.autocorr(lag) if len(x) >= lag + 2 else 0
        )

    # ── Z-Scores (4 features) ──
    for p in [20, 50, 100, 200]:
        roll_mean = close.rolling(p).mean()
        roll_std = close.rolling(p).std()
        feat[f"zscore_{p}"] = (close - roll_mean) / (roll_std + 1e-10)

    # ── Percentile Ranks (3 features) ──
    for p in [20, 60, 252]:
        feat[f"percentile_rank_{p}d"] = close.rolling(p).apply(
            lambda x: (x < x.iloc[-1]).sum() / len(x) * 100 if len(x) == p else 50
        )

    # ── Max Drawdown (2 features) ──
    for p in [20, 60]:
        rolling_max = close.rolling(p).max()
        feat[f"max_drawdown_{p}d"] = (close - rolling_max) / (rolling_max + 1e-10)

    # ── Up/Down Ratios (2 features) ──
    for p in [10, 20]:
        feat[f"up_ratio_{p}d"] = (close.diff() > 0).rolling(p).mean()

    # ── Gap Features (2 features) ──
    gap = (open_ - close.shift(1)) / (close.shift(1) + 1e-10)
    feat["avg_gap_20d"] = gap.rolling(20).mean()
    feat["gap_frequency_20d"] = (gap.abs() > 0.01).rolling(20).mean()

    # ── Calendar Features (5 features) ──
    dates = pd.to_datetime(df.index)
    feat["day_of_week"] = dates.dayofweek / 4
    feat["month_sin"] = np.sin(2 * np.pi * dates.month / 12)
    feat["month_cos"] = np.cos(2 * np.pi * dates.month / 12)
    feat["is_quarter_end"] = dates.month.isin([3, 6, 9, 12]).astype(float)
    feat["is_opex_week"] = ((dates.day >= 15) & (dates.day <= 21)).astype(float)

    # ── Trend Strength (3 features) ──
    feat["price_slope_20"] = close.rolling(20).apply(
        lambda x: np.polyfit(range(len(x)), x / x.iloc[0], 1)[0] if len(x) == 20 else 0
    )
    feat["price_slope_50"] = close.rolling(50).apply(
        lambda x: np.polyfit(range(len(x)), x / x.iloc[0], 1)[0] if len(x) == 50 else 0
    )
    tenkan = (high.rolling(9).max() + low.rolling(9).min()) / 2
    kijun = (high.rolling(26).max() + low.rolling(26).min()) / 2
    feat["ichimoku_tk_cross"] = (tenkan > kijun).astype(float)

    # ── Volatility Regime (2 features) ──
    feat["vol_regime_ratio"] = feat["volatility_20d"] / (feat["volatility_60d"] + 1e-10)
    feat["vol_expanding"] = (feat["volatility_20d"] > feat["volatility_60d"]).astype(float)

    # ── Macro Features (if available) ──
    if macro_df is not None:
        # VIX level and change
        if "^VIX" in macro_df.columns:
            vix = macro_df["^VIX"].reindex(df.index, method="ffill")
            feat["vix_level"] = vix
            feat["vix_change_5d"] = vix.pct_change(5)
            feat["vix_zscore_20"] = (vix - vix.rolling(20).mean()) / (vix.rolling(20).std() + 1e-10)

        # VIX term structure (contango/backwardation)
        if "^VIX" in macro_df.columns and "^VIX3M" in macro_df.columns:
            vix = macro_df["^VIX"].reindex(df.index, method="ffill")
            vix3m = macro_df["^VIX3M"].reindex(df.index, method="ffill")
            feat["vix_term_structure"] = (vix3m - vix) / (vix + 1e-10)

        # Treasury yield
        if "^TNX" in macro_df.columns:
            tnx = macro_df["^TNX"].reindex(df.index, method="ffill")
            feat["treasury_10y"] = tnx
            feat["treasury_change_20d"] = tnx.diff(20)

        # USD Index
        if "DX-Y.NYB" in macro_df.columns:
            dxy = macro_df["DX-Y.NYB"].reindex(df.index, method="ffill")
            feat["usd_index"] = dxy
            feat["usd_change_20d"] = dxy.pct_change(20)

        # Gold
        if "GC=F" in macro_df.columns:
            gold = macro_df["GC=F"].reindex(df.index, method="ffill")
            feat["gold_change_20d"] = gold.pct_change(20)

        # Oil
        if "CL=F" in macro_df.columns:
            oil = macro_df["CL=F"].reindex(df.index, method="ffill")
            feat["oil_change_20d"] = oil.pct_change(20)

    # Clean up
    feat = feat.replace([np.inf, -np.inf], np.nan)
    feat = feat.fillna(0)

    return feat


print(f"[OK] Feature engineering function defined")

In [ ]:
# Cell 4: Data Download & Preparation
# Downloads 10 years of data for all 80+ stocks + macro indicators

import time

def download_macro_data(tickers: List[str], period: str = "10y") -> pd.DataFrame:
    """Download macro indicator data (VIX, yields, USD, Gold, Oil)."""
    macro_df = pd.DataFrame()
    for ticker in tickers:
        try:
            df = yf.download(ticker, period=period, interval="1d", progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            if len(df) > 0:
                macro_df[ticker] = df["Close"].squeeze()
                print(f"  Macro {ticker}: {len(df)} days")
        except Exception as e:
            print(f"  Macro {ticker}: FAILED ({e})")
    return macro_df


def download_and_prepare_data(
    symbols: List[str],
    macro_tickers: List[str],
    lookback: int,
    horizon: int,
    data_years: int = 10,
) -> Tuple[np.ndarray, np.ndarray, List[str], dict, dict]:
    """
    Download data for all stocks, compute features with macro indicators,
    create training samples using walk-forward approach.

    Returns:
        X: (num_samples, lookback, num_features)
        y: (num_samples, horizon) — future returns
        feature_names: list of feature column names
        normalization_stats: per-stock mean/std for inference
        stock_sample_indices: dict mapping symbol -> (start_idx, end_idx) in X/y
    """
    # Download macro data first
    print("  Downloading macro indicators...")
    macro_df = download_macro_data(macro_tickers, period=f"{data_years}y")
    print(f"  Macro data: {len(macro_df)} days, {len(macro_df.columns)} indicators\n")

    all_X = []
    all_y = []
    feature_names = None
    normalization_stats = {}
    stock_sample_indices = {}
    failed = []

    for i, sym in enumerate(symbols):
        print(f"  [{i+1}/{len(symbols)}] {sym}...", end=" ", flush=True)
        try:
            df = yf.download(sym, period=f"{data_years}y", interval="1d", progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            if len(df) < lookback + horizon + 252:
                print(f"skipped (only {len(df)} days)")
                failed.append(sym)
                continue

            features = compute_features(df, macro_df)
            closes = df["Close"].values.flatten()

            if feature_names is None:
                feature_names = list(features.columns)
                print(f"{len(feature_names)} features, {len(df)} days")
            else:
                print(f"{len(df)} days")

            # Z-score normalize per stock
            feat_values = features.values
            feat_mean = np.nanmean(feat_values, axis=0, keepdims=True)
            feat_std = np.nanstd(feat_values, axis=0, keepdims=True) + 1e-10
            feat_norm = (feat_values - feat_mean) / feat_std
            feat_norm = np.clip(feat_norm, -5, 5)

            normalization_stats[sym] = {
                "mean": feat_mean.flatten().tolist(),
                "std": feat_std.flatten().tolist(),
            }

            # Track start index for this stock's samples
            start_idx = len(all_X)

            # Create sliding window samples
            for j in range(lookback, len(feat_norm) - horizon):
                X_sample = feat_norm[j - lookback:j]
                current_price = closes[j]
                future_prices = closes[j + 1:j + horizon + 1]
                if len(future_prices) == horizon and current_price > 0:
                    y_sample = (future_prices - current_price) / current_price
                    all_X.append(X_sample)
                    all_y.append(y_sample)

            end_idx = len(all_X)
            stock_sample_indices[sym] = (start_idx, end_idx)

            # Rate limiting for Yahoo Finance
            if (i + 1) % 5 == 0:
                time.sleep(1)

        except Exception as e:
            print(f"FAILED ({e})")
            failed.append(sym)
            continue

    X = np.array(all_X, dtype=np.float32)
    y = np.array(all_y, dtype=np.float32)

    print(f"\n[DATA] {X.shape[0]:,} total samples from {len(symbols) - len(failed)} stocks")
    print(f"[DATA] {X.shape[2]} features, lookback={X.shape[1]}, horizon={y.shape[1]}")
    if failed:
        print(f"[DATA] Failed: {', '.join(failed)}")

    return X, y, feature_names or [], normalization_stats, stock_sample_indices


print("[1/7] Downloading market data and computing features...")
print(f"  {len(SCREENER_SYMBOLS)} stocks × {DATA_YEARS} years + {len(MACRO_TICKERS)} macro indicators\n")
X, y, feature_names, norm_stats, stock_sample_indices = download_and_prepare_data(
    SCREENER_SYMBOLS, MACRO_TICKERS, LOOKBACK_WINDOW, FORECAST_HORIZON, DATA_YEARS
)
print(f"\n[DATA] Features ({len(feature_names)}): {feature_names[:10]}...")
print(f"[DATA] Per-stock sample counts:")
for sym, (s, e) in sorted(stock_sample_indices.items(), key=lambda x: x[1][1]-x[1][0], reverse=True)[:10]:
    print(f"  {sym}: {e-s:,} samples")
print(f"  ... and {len(stock_sample_indices)-10} more stocks")

In [ ]:
# Cell 5: iTransformer Model Architecture
# Following the original paper: RevIN → shared embedding → cross-variate attention → shared projection

import torch
import torch.nn as nn
import math


class RevIN(nn.Module):
    """
    Reversible Instance Normalization (Kim et al., ICLR 2022).
    Per-window normalization for non-stationary financial data.
    """
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if affine:
            self.affine_weight = nn.Parameter(torch.ones(num_features))
            self.affine_bias = nn.Parameter(torch.zeros(num_features))

    def forward(self, x: torch.Tensor, mode: str = "norm") -> torch.Tensor:
        if mode == "norm":
            self._mean = x.mean(dim=1, keepdim=True).detach()
            self._std = (x.std(dim=1, keepdim=True) + self.eps).detach()
            x = (x - self._mean) / self._std
            if self.affine:
                x = x * self.affine_weight + self.affine_bias
            return x
        else:
            if self.affine:
                x = (x - self.affine_bias) / (self.affine_weight + self.eps)
            return x


class iTransformer(nn.Module):
    """
    Inverted Transformer for Time-Series Forecasting (ICLR 2024, Liu et al.).

    1. RevIN normalization (per-window)
    2. INVERT: transpose to (batch, num_variates, lookback)
    3. Shared embedding: Linear(lookback -> d_model) per variate
    4. Learnable variate tokens
    5. Transformer encoder: self-attention across variates
    6. Shared projection: Linear(d_model -> forecast_horizon)
    7. Weighted aggregation across variates
    """

    def __init__(
        self,
        num_variates: int,
        lookback: int,
        forecast_horizon: int,
        d_model: int = 128,
        n_heads: int = 8,
        n_layers: int = 3,
        d_ff: int = 256,
        dropout: float = 0.2,
        use_norm: bool = True,
    ):
        super().__init__()
        self.num_variates = num_variates
        self.lookback = lookback
        self.forecast_horizon = forecast_horizon
        self.d_model = d_model
        self.use_norm = use_norm

        if use_norm:
            self.revin = RevIN(num_variates)

        self.variate_embedding = nn.Linear(lookback, d_model)
        self.variate_tokens = nn.Parameter(torch.randn(1, num_variates, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.projection = nn.Linear(d_model, forecast_horizon, bias=True)
        self.agg_weights = nn.Parameter(torch.ones(num_variates) / num_variates)

        self.output_head = nn.Sequential(
            nn.LayerNorm(forecast_horizon),
            nn.Linear(forecast_horizon, forecast_horizon),
            nn.Tanh(),
        )
        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if p.dim() > 1 and 'revin' not in name:
                nn.init.xavier_uniform_(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_norm:
            x = self.revin(x, mode="norm")
        x = x.transpose(1, 2)
        tokens = self.variate_embedding(x)
        tokens = tokens + self.variate_tokens
        encoded = self.encoder(tokens)
        per_variate_forecast = self.projection(encoded)
        weights = torch.softmax(self.agg_weights, dim=0)
        forecast = torch.einsum('bvh,v->bh', per_variate_forecast, weights)
        forecast = self.output_head(forecast)
        return forecast

    def get_variate_importance(self) -> np.ndarray:
        with torch.no_grad():
            return torch.softmax(self.agg_weights, dim=0).cpu().numpy()


# Instantiate model
model = iTransformer(
    num_variates=X.shape[2],
    lookback=X.shape[1],
    forecast_horizon=y.shape[1],
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
    use_norm=True,
).to(device)

param_count = sum(p.numel() for p in model.parameters())
print(f"[MODEL] iTransformer — {param_count:,} parameters")
print(f"  Variates (tokens): {X.shape[2]}, Lookback: {X.shape[1]}d, Forecast: {y.shape[1]}d")
print(f"  d_model={D_MODEL}, layers={N_LAYERS}, heads={N_HEADS}, RevIN=enabled")

In [ ]:
# Cell 6: Training Loop with Walk-Forward Validation

from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


def train_model(X, y, model, config):
    """
    Train with walk-forward validation (70/15/15 chronological split).
    Huber loss, LR warmup + cosine decay, directional accuracy tracking.
    """
    n = X.shape[0]
    train_end = int(n * config["train_split"])
    val_end = int(n * config["val_split"])

    X_train, y_train = X[:train_end], y[:train_end]
    X_val, y_val = X[train_end:val_end], y[train_end:val_end]
    X_test, y_test = X[val_end:], y[val_end:]

    print(f"  Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
    val_ds = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val))
    test_ds = TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test))

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"])
    test_loader = DataLoader(test_ds, batch_size=config["batch_size"])

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config["lr"],
        weight_decay=config["weight_decay"], betas=(0.9, 0.999),
    )

    def lr_lambda(epoch):
        if epoch < config["warmup_epochs"]:
            return (epoch + 1) / config["warmup_epochs"]
        progress = (epoch - config["warmup_epochs"]) / max(1, config["epochs"] - config["warmup_epochs"])
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion = nn.HuberLoss(delta=0.02)

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "val_dir_acc": [], "lr": []}

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        all_pred, all_true = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                pred = model(X_batch)
                val_loss += criterion(pred, y_batch).item()
                all_pred.append(pred.cpu())
                all_true.append(y_batch.cpu())
        val_loss /= len(val_loader)

        preds = torch.cat(all_pred)
        trues = torch.cat(all_true)
        dir_acc = ((preds[:, -1] > 0) == (trues[:, -1] > 0)).float().mean().item() * 100

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_dir_acc"].append(dir_acc)
        history["lr"].append(current_lr)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            marker = "*best*" if patience_counter == 0 else f"(patience {patience_counter}/{config['patience']})"
            print(f"  Epoch {epoch+1:3d}/{config['epochs']} — "
                  f"train: {train_loss:.6f}, val: {val_loss:.6f}, "
                  f"dir_acc: {dir_acc:.1f}%, lr: {current_lr:.2e} {marker}")

        if patience_counter >= config["patience"]:
            print(f"  [EARLY STOP] No improvement for {config['patience']} epochs.")
            break

    if best_state:
        model.load_state_dict(best_state)
    model = model.to(device)

    # Test set evaluation
    model.eval()
    test_preds, test_trues = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            pred = model(X_batch.to(device))
            test_preds.append(pred.cpu())
            test_trues.append(y_batch)

    test_preds = torch.cat(test_preds)
    test_trues = torch.cat(test_trues)

    test_mse = ((test_preds - test_trues) ** 2).mean().item()
    test_mae = (test_preds - test_trues).abs().mean().item()

    # Per-horizon directional accuracy
    horizon_dir_acc = []
    for h in range(test_preds.shape[1]):
        acc = ((test_preds[:, h] > 0) == (test_trues[:, h] > 0)).float().mean().item() * 100
        horizon_dir_acc.append(acc)

    history["test_metrics"] = {
        "mse": test_mse, "mae": test_mae,
        "dir_acc_60d": float(horizon_dir_acc[-1]),
        "dir_acc_30d": float(horizon_dir_acc[29]) if len(horizon_dir_acc) > 29 else float(horizon_dir_acc[-1]),
        "dir_acc_7d": float(horizon_dir_acc[6]) if len(horizon_dir_acc) > 6 else float(horizon_dir_acc[0]),
        "dir_acc_14d": float(horizon_dir_acc[13]) if len(horizon_dir_acc) > 13 else float(horizon_dir_acc[0]),
        "horizon_dir_acc": horizon_dir_acc,
    }

    print(f"\n[TEST RESULTS]")
    print(f"  MSE: {test_mse:.6f}, MAE: {test_mae:.6f}")
    print(f"  7-Day Dir Acc:  {history['test_metrics']['dir_acc_7d']:.1f}%")
    print(f"  14-Day Dir Acc: {history['test_metrics']['dir_acc_14d']:.1f}%")
    print(f"  30-Day Dir Acc: {history['test_metrics']['dir_acc_30d']:.1f}%")
    print(f"  60-Day Dir Acc: {history['test_metrics']['dir_acc_60d']:.1f}%")

    return model, history


print("[2/7] Training iTransformer...")
config = {
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
    "patience": PATIENCE, "train_split": TRAIN_SPLIT, "val_split": VAL_SPLIT,
}
model, history = train_model(X, y, model, config)

In [ ]:
# Cell 7: Training Visualization

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("iTransformer Universal Model — Training Results", fontsize=14, fontweight="bold")

ax1 = axes[0, 0]
ax1.plot(history["train_loss"], label="Train", alpha=0.8)
ax1.plot(history["val_loss"], label="Validation", alpha=0.8)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Huber Loss")
ax1.set_title("Training & Validation Loss"); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(history["val_dir_acc"], label="Val Dir Acc (60d)", color="green", alpha=0.8)
ax2.axhline(y=50, color="red", linestyle="--", alpha=0.5, label="Random (50%)")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Directional Accuracy"); ax2.legend(); ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
test_metrics = history["test_metrics"]
days = list(range(1, len(test_metrics["horizon_dir_acc"]) + 1))
ax3.bar(days, test_metrics["horizon_dir_acc"], alpha=0.7, color="steelblue")
ax3.axhline(y=50, color="red", linestyle="--", alpha=0.5, label="Random")
# Mark DTE preset boundaries
for d, label in [(7, "7d"), (14, "14d"), (30, "30d"), (45, "45d"), (60, "60d")]:
    if d <= len(days):
        ax3.axvline(x=d, color="orange", linestyle=":", alpha=0.5)
        ax3.text(d, max(test_metrics["horizon_dir_acc"]) + 1, label, ha="center", fontsize=7, color="orange")
ax3.set_xlabel("Forecast Day"); ax3.set_ylabel("Dir Accuracy (%)")
ax3.set_title("Test: Per-Horizon Directional Accuracy"); ax3.legend(); ax3.grid(True, alpha=0.3)

ax4 = axes[1, 1]
importance = model.get_variate_importance()
top_idx = np.argsort(importance)[-20:]
top_names = [feature_names[i] for i in top_idx]
top_weights = importance[top_idx]
ax4.barh(top_names, top_weights, color="coral", alpha=0.8)
ax4.set_xlabel("Aggregation Weight"); ax4.set_title("Top 20 Feature Importance"); ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n[SUMMARY]")
print(f"  Best Val Loss:      {min(history['val_loss']):.6f}")
print(f"  Test MSE:           {test_metrics['mse']:.6f}")
print(f"  Test 7d Dir Acc:    {test_metrics['dir_acc_7d']:.1f}%")
print(f"  Test 30d Dir Acc:   {test_metrics['dir_acc_30d']:.1f}%")
print(f"  Test 60d Dir Acc:   {test_metrics['dir_acc_60d']:.1f}%")
print(f"  Epochs trained:     {len(history['train_loss'])}")

In [ ]:
# Cell 8: ONNX Export & Verification

import json
import onnx
import onnxruntime as ort

print("[3/7] Exporting to ONNX...")

# Export to ONNX
model.eval()
model_cpu = model.cpu()
dummy_input = torch.randn(1, LOOKBACK_WINDOW, len(feature_names))

torch.onnx.export(
    model_cpu,
    dummy_input,
    "itransformer.onnx",
    input_names=["features"],
    output_names=["forecast"],
    dynamic_axes={
        "features": {0: "batch_size"},
        "forecast": {0: "batch_size"},
    },
    opset_version=17,
    do_constant_folding=True,
)

# Consolidate external data into a single self-contained ONNX file.
# PyTorch may split tensors into a separate .data file (external data format),
# which onnxruntime-web cannot load via MountedFiles in the browser.
# Loading with load_external_data=True and re-saving ensures all weights are
# embedded directly in the .onnx protobuf.
onnx_model = onnx.load("itransformer.onnx", load_external_data=True)
onnx.save_model(
    onnx_model,
    "itransformer.onnx",
    save_as_external_data=False,
)
print("  Consolidated external data into single ONNX file")

# Remove leftover .data file if it exists
if os.path.exists("itransformer.onnx.data"):
    os.remove("itransformer.onnx.data")
    print("  Removed leftover itransformer.onnx.data")

# Verify ONNX model
onnx_model = onnx.load("itransformer.onnx")
onnx.checker.check_model(onnx_model)

# Test with ONNX Runtime
session = ort.InferenceSession("itransformer.onnx")
test_input = np.random.randn(1, LOOKBACK_WINDOW, len(feature_names)).astype(np.float32)
onnx_output = session.run(None, {"features": test_input})[0]

# Compare PyTorch vs ONNX output
with torch.no_grad():
    torch_output = model_cpu(torch.FloatTensor(test_input)).numpy()

max_diff = np.abs(torch_output - onnx_output).max()

onnx_size = os.path.getsize("itransformer.onnx") / (1024 * 1024)
print(f"  ONNX file: itransformer.onnx ({onnx_size:.1f} MB)")
print(f"  Output shape: {onnx_output.shape}")
print(f"  PyTorch vs ONNX max diff: {max_diff:.8f} {'(OK)' if max_diff < 1e-4 else '(WARNING: large diff)'}")

# Sanity check: model with 427K params should be ~1.7 MB, not 277 KB
if onnx_size < 0.5:
    print(f"  [WARNING] ONNX file is only {onnx_size:.1f} MB — weights may not be embedded!")
    print(f"  Expected ~{sum(p.numel() for p in model.parameters()) * 4 / 1024 / 1024:.1f} MB for {sum(p.numel() for p in model.parameters()):,} float32 params")

# Move model back to GPU for any further use
model = model.to(device)

# Save model config (for website to load and reconstruct)
model_config = {
    "model_name": "putstrike-itransformer",
    "version": "3.0",
    "paper": "Inverted Transformers Are Effective for Time Series Forecasting (ICLR 2024)",
    "feature_names": feature_names,
    "num_features": len(feature_names),
    "lookback": LOOKBACK_WINDOW,
    "forecast_horizon": FORECAST_HORIZON,
    "architecture": {
        "type": "iTransformer",
        "d_model": D_MODEL,
        "n_layers": N_LAYERS,
        "n_heads": N_HEADS,
        "d_ff": D_FF,
        "dropout": DROPOUT,
        "use_revin": True,
        "parameters": sum(p.numel() for p in model.parameters()),
    },
    "training": {
        "symbols": SCREENER_SYMBOLS,
        "num_stocks": len(SCREENER_SYMBOLS),
        "macro_tickers": MACRO_TICKERS,
        "total_samples": int(X.shape[0]),
        "data_years": DATA_YEARS,
        "epochs_trained": len(history["train_loss"]),
        "best_val_loss": float(min(history["val_loss"])),
        "loss_function": "HuberLoss(delta=0.02)",
        "optimizer": "AdamW",
        "lr_schedule": "warmup + cosine decay",
        "validation": "walk-forward (70/15/15)",
        "batch_size": BATCH_SIZE,
    },
    "test_metrics": {
        "mse": float(test_metrics["mse"]),
        "mae": float(test_metrics["mae"]),
        "dir_acc_7d": float(test_metrics["dir_acc_7d"]),
        "dir_acc_14d": float(test_metrics["dir_acc_14d"]),
        "dir_acc_30d": float(test_metrics["dir_acc_30d"]),
        "dir_acc_60d": float(test_metrics["dir_acc_60d"]),
        "horizon_dir_acc": test_metrics["horizon_dir_acc"],
    },
    "normalization_stats": norm_stats,
    "onnx_size_mb": round(onnx_size, 2),
}

with open("model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

print(f"  Config: model_config.json")
print(f"  {len(feature_names)} features, {len(norm_stats)} stock normalization profiles")

In [ ]:
# Cell 8b: Per-Stock Individual Model Training & ONNX Export
# Trains one iTransformer model per stock using only that stock's data,
# exports each to ONNX, and collects per-stock test metrics.

import gc

print("[3b/7] Training per-stock individual models...")
print(f"  {len(stock_sample_indices)} stocks to train\n")

# Per-stock training config — fewer epochs, lower patience (less data per stock)
PER_STOCK_EPOCHS = 60
PER_STOCK_PATIENCE = 10
PER_STOCK_LR = 3e-4
PER_STOCK_BATCH_SIZE = 32
MIN_SAMPLES = 500  # Skip stocks with too few samples

os.makedirs("per_stock", exist_ok=True)

per_stock_metrics = {}
per_stock_failed = []

for idx, (sym, (start_idx, end_idx)) in enumerate(stock_sample_indices.items()):
    n_samples = end_idx - start_idx
    print(f"  [{idx+1}/{len(stock_sample_indices)}] {sym} ({n_samples:,} samples)...", end=" ", flush=True)

    if n_samples < MIN_SAMPLES:
        print(f"skipped (< {MIN_SAMPLES} samples)")
        per_stock_failed.append(sym)
        continue

    try:
        # Extract this stock's data
        X_stock = X[start_idx:end_idx]
        y_stock = y[start_idx:end_idx]

        # Walk-forward split (same 70/15/15 as universal)
        n = X_stock.shape[0]
        train_end = int(n * TRAIN_SPLIT)
        val_end = int(n * VAL_SPLIT)

        X_train_s = X_stock[:train_end]
        y_train_s = y_stock[:train_end]
        X_val_s = X_stock[train_end:val_end]
        y_val_s = y_stock[train_end:val_end]
        X_test_s = X_stock[val_end:]
        y_test_s = y_stock[val_end:]

        if len(X_val_s) < 10 or len(X_test_s) < 10:
            print("skipped (insufficient val/test data)")
            per_stock_failed.append(sym)
            continue

        # Create per-stock model (same architecture as universal)
        stock_model = iTransformer(
            num_variates=X_stock.shape[2],
            lookback=X_stock.shape[1],
            forecast_horizon=y_stock.shape[1],
            d_model=D_MODEL,
            n_layers=N_LAYERS,
            n_heads=N_HEADS,
            d_ff=D_FF,
            dropout=DROPOUT,
            use_norm=True,
        ).to(device)

        train_ds = TensorDataset(torch.FloatTensor(X_train_s), torch.FloatTensor(y_train_s))
        val_ds = TensorDataset(torch.FloatTensor(X_val_s), torch.FloatTensor(y_val_s))
        test_ds = TensorDataset(torch.FloatTensor(X_test_s), torch.FloatTensor(y_test_s))

        train_loader = DataLoader(train_ds, batch_size=PER_STOCK_BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=PER_STOCK_BATCH_SIZE)
        test_loader = DataLoader(test_ds, batch_size=PER_STOCK_BATCH_SIZE)

        optimizer = torch.optim.AdamW(
            stock_model.parameters(), lr=PER_STOCK_LR,
            weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999),
        )

        def lr_lambda_stock(epoch):
            if epoch < 3:
                return (epoch + 1) / 3
            progress = (epoch - 3) / max(1, PER_STOCK_EPOCHS - 3)
            return 0.5 * (1 + math.cos(math.pi * progress))

        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda_stock)
        criterion = nn.HuberLoss(delta=0.02)

        best_val_loss = float("inf")
        best_state = None
        patience_counter = 0

        for epoch in range(PER_STOCK_EPOCHS):
            stock_model.train()
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                pred = stock_model(X_batch)
                loss = criterion(pred, y_batch)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(stock_model.parameters(), 1.0)
                optimizer.step()

            stock_model.eval()
            val_loss = 0
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    pred = stock_model(X_batch)
                    val_loss += criterion(pred, y_batch).item()
            val_loss /= len(val_loader)

            scheduler.step()

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in stock_model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= PER_STOCK_PATIENCE:
                break

        # Load best weights
        if best_state:
            stock_model.load_state_dict(best_state)
        stock_model = stock_model.to(device)

        # Test evaluation
        stock_model.eval()
        test_preds, test_trues = [], []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                pred = stock_model(X_batch.to(device))
                test_preds.append(pred.cpu())
                test_trues.append(y_batch)

        test_preds_t = torch.cat(test_preds)
        test_trues_t = torch.cat(test_trues)

        stock_mse = ((test_preds_t - test_trues_t) ** 2).mean().item()
        stock_mae = (test_preds_t - test_trues_t).abs().mean().item()

        # Per-horizon directional accuracy
        horizon_dir_acc = []
        for h in range(test_preds_t.shape[1]):
            acc = ((test_preds_t[:, h] > 0) == (test_trues_t[:, h] > 0)).float().mean().item() * 100
            horizon_dir_acc.append(acc)

        dir_7d = horizon_dir_acc[6] if len(horizon_dir_acc) > 6 else horizon_dir_acc[0]
        dir_14d = horizon_dir_acc[13] if len(horizon_dir_acc) > 13 else horizon_dir_acc[0]
        dir_30d = horizon_dir_acc[29] if len(horizon_dir_acc) > 29 else horizon_dir_acc[-1]
        dir_60d = horizon_dir_acc[-1]

        # Export to ONNX
        stock_model_cpu = stock_model.cpu()
        stock_model_cpu.eval()
        dummy = torch.randn(1, LOOKBACK_WINDOW, len(feature_names))

        onnx_path = f"per_stock/{sym}.onnx"
        torch.onnx.export(
            stock_model_cpu, dummy, onnx_path,
            input_names=["features"], output_names=["forecast"],
            dynamic_axes={"features": {0: "batch_size"}, "forecast": {0: "batch_size"}},
            opset_version=17, do_constant_folding=True,
        )

        # Consolidate external data into single file
        onnx_m = onnx.load(onnx_path, load_external_data=True)
        onnx.save_model(onnx_m, onnx_path, save_as_external_data=False)
        data_path = f"{onnx_path}.data"
        if os.path.exists(data_path):
            os.remove(data_path)

        onnx_size_s = os.path.getsize(onnx_path) / (1024 * 1024)

        per_stock_metrics[sym] = {
            "mse": stock_mse, "mae": stock_mae,
            "dir_acc_7d": dir_7d, "dir_acc_14d": dir_14d,
            "dir_acc_30d": dir_30d, "dir_acc_60d": dir_60d,
            "horizon_dir_acc": horizon_dir_acc,
            "num_samples": n_samples,
            "epochs_trained": epoch + 1,
            "best_val_loss": best_val_loss,
            "onnx_size_mb": round(onnx_size_s, 2),
        }

        print(f"dir_30d={dir_30d:.1f}%, {epoch+1} epochs, {onnx_size_s:.1f}MB")

        # Clean up GPU memory
        del stock_model, stock_model_cpu, optimizer, scheduler
        del train_ds, val_ds, test_ds, train_loader, val_loader, test_loader
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        gc.collect()

    except Exception as e:
        print(f"FAILED ({e})")
        per_stock_failed.append(sym)
        gc.collect()
        continue

# Save per-stock config
per_stock_config = {
    "model_name": "putstrike-itransformer-per-stock",
    "version": "3.0",
    "num_features": len(feature_names),
    "lookback": LOOKBACK_WINDOW,
    "forecast_horizon": FORECAST_HORIZON,
    "architecture": {
        "type": "iTransformer",
        "d_model": D_MODEL,
        "n_layers": N_LAYERS,
        "n_heads": N_HEADS,
        "d_ff": D_FF,
        "dropout": DROPOUT,
        "use_revin": True,
        "parameters": sum(p.numel() for p in model.parameters()),
    },
    "per_stock_metrics": per_stock_metrics,
    "failed_stocks": per_stock_failed,
    "training_config": {
        "epochs": PER_STOCK_EPOCHS,
        "patience": PER_STOCK_PATIENCE,
        "batch_size": PER_STOCK_BATCH_SIZE,
        "lr": PER_STOCK_LR,
        "min_samples": MIN_SAMPLES,
    },
}

with open("per_stock/per_stock_config.json", "w") as f:
    json.dump(per_stock_config, f, indent=2)

print(f"\n[PER-STOCK] Trained {len(per_stock_metrics)} models, {len(per_stock_failed)} failed/skipped")
if per_stock_metrics:
    avg_dir30 = np.mean([m["dir_acc_30d"] for m in per_stock_metrics.values()])
    print(f"[PER-STOCK] Average 30d dir accuracy: {avg_dir30:.1f}%")
    print(f"[PER-STOCK] Universal 30d dir accuracy: {test_metrics['dir_acc_30d']:.1f}%")

In [ ]:
# Cell 9: Push to HuggingFace Hub
from huggingface_hub import HfApi, create_repo
from google.colab import userdata
import glob as glob_module

print("[4/7] Pushing to HuggingFace Hub...")

# Load secrets from Google Colab
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    HF_REPO_ID = userdata.get('HF_REPO_ID')
except userdata.SecretNotFoundError as e:
    print(f"  [SKIP] Missing Colab secret: {e}")
    print("  Add secrets via: Secrets (key) panel in the left sidebar")
    HF_TOKEN = None
    HF_REPO_ID = None

if not HF_TOKEN or not HF_REPO_ID:
    print("  [SKIP] HF_TOKEN and HF_REPO_ID secrets required to push to HuggingFace Hub")
    print("  Get your token: https://huggingface.co/settings/tokens")
else:
    api = HfApi(token=HF_TOKEN)

    # Create repo if it doesn't exist
    try:
        create_repo(HF_REPO_ID, token=HF_TOKEN, repo_type="model", exist_ok=True)
        print(f"  Repo: https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"  Repo creation: {e}")

    # ── Upload Universal Model ──
    api.upload_file(
        path_or_fileobj="itransformer.onnx",
        path_in_repo="itransformer.onnx",
        repo_id=HF_REPO_ID,
        token=HF_TOKEN,
    )
    print(f"  Uploaded: itransformer.onnx ({onnx_size:.1f} MB)")

    # Upload external data file if it exists (safety net)
    if os.path.exists("itransformer.onnx.data"):
        ext_size = os.path.getsize("itransformer.onnx.data") / (1024 * 1024)
        api.upload_file(
            path_or_fileobj="itransformer.onnx.data",
            path_in_repo="itransformer.onnx.data",
            repo_id=HF_REPO_ID,
            token=HF_TOKEN,
        )
        print(f"  Uploaded: itransformer.onnx.data ({ext_size:.1f} MB)")

    # Upload universal config
    api.upload_file(
        path_or_fileobj="model_config.json",
        path_in_repo="model_config.json",
        repo_id=HF_REPO_ID,
        token=HF_TOKEN,
    )
    print(f"  Uploaded: model_config.json")

    # ── Upload Per-Stock Models ──
    per_stock_onnx_files = sorted(glob_module.glob("per_stock/*.onnx"))
    if per_stock_onnx_files:
        print(f"\n  Uploading {len(per_stock_onnx_files)} per-stock models...")
        for onnx_file in per_stock_onnx_files:
            sym_name = os.path.basename(onnx_file).replace(".onnx", "")
            file_size = os.path.getsize(onnx_file) / (1024 * 1024)
            api.upload_file(
                path_or_fileobj=onnx_file,
                path_in_repo=f"per_stock/{sym_name}.onnx",
                repo_id=HF_REPO_ID,
                token=HF_TOKEN,
            )
            print(f"    {sym_name}.onnx ({file_size:.1f} MB)")

        # Upload per-stock config
        if os.path.exists("per_stock/per_stock_config.json"):
            api.upload_file(
                path_or_fileobj="per_stock/per_stock_config.json",
                path_in_repo="per_stock/per_stock_config.json",
                repo_id=HF_REPO_ID,
                token=HF_TOKEN,
            )
            print(f"  Uploaded: per_stock/per_stock_config.json")

    # ── Upload README ──
    per_stock_section = ""
    if per_stock_metrics:
        avg_dir30 = np.mean([m["dir_acc_30d"] for m in per_stock_metrics.values()])
        per_stock_section = f"""

## Per-Stock Models

In addition to the universal model, **{len(per_stock_metrics)} individual per-stock models** are available
under `per_stock/{{SYMBOL}}.onnx`. Each is trained exclusively on that stock's data.

- **Average 30-day directional accuracy**: {avg_dir30:.1f}%
- **Architecture**: Same as universal (iTransformer, {sum(p.numel() for p in model.parameters()):,} params)
- **Training**: Per-stock walk-forward validation (70/15/15)

The website automatically uses the per-stock model when available, falling back to the universal model.
"""

    readme_content = f"""---
tags:
  - time-series
  - finance
  - itransformer
  - onnx
  - stock-prediction
license: mit
---

# PutStrike iTransformer — Universal + Per-Stock Forecasting Models

**iTransformer** (ICLR 2024) trained on {len(SCREENER_SYMBOLS)} stocks for {FORECAST_HORIZON}-day price forecasting.

## Universal Model

- **Architecture**: iTransformer with RevIN ({sum(p.numel() for p in model.parameters()):,} parameters)
- **Input**: {LOOKBACK_WINDOW} days x {len(feature_names)} features (OHLCV technicals + macro)
- **Output**: {FORECAST_HORIZON}-day forward return forecast
- **Training**: {X.shape[0]:,} samples from all stocks, walk-forward validation (70/15/15)
- **Loss**: HuberLoss(delta=0.02)

### Test Metrics (Universal)

| Horizon | Directional Accuracy |
|---------|---------------------|
| 7-day   | {test_metrics['dir_acc_7d']:.1f}% |
| 14-day  | {test_metrics['dir_acc_14d']:.1f}% |
| 30-day  | {test_metrics['dir_acc_30d']:.1f}% |
| 60-day  | {test_metrics['dir_acc_60d']:.1f}% |
{per_stock_section}
## Usage

```python
import onnxruntime as ort
import numpy as np

# Universal model
session = ort.InferenceSession("itransformer.onnx")

# Or per-stock model
session = ort.InferenceSession("per_stock/AAPL.onnx")

# features shape: (1, {LOOKBACK_WINDOW}, {len(feature_names)})
output = session.run(None, {{"features": features}})[0]
# output shape: (1, {FORECAST_HORIZON}) — predicted returns
```

## Disclaimer

This model is for research and educational purposes only. Not financial advice.
"""

    api.upload_file(
        path_or_fileobj=readme_content.encode(),
        path_in_repo="README.md",
        repo_id=HF_REPO_ID,
        token=HF_TOKEN,
    )
    print(f"  Uploaded: README.md")

    # Upload training visualization
    if os.path.exists("training_results.png"):
        api.upload_file(
            path_or_fileobj="training_results.png",
            path_in_repo="training_results.png",
            repo_id=HF_REPO_ID,
            token=HF_TOKEN,
        )
        print(f"  Uploaded: training_results.png")

    print(f"\n  Model available at: https://huggingface.co/{HF_REPO_ID}")
    print(f"  Universal: https://huggingface.co/{HF_REPO_ID}/resolve/main/itransformer.onnx")
    if per_stock_onnx_files:
        print(f"  Per-stock: https://huggingface.co/{HF_REPO_ID}/resolve/main/per_stock/{{SYMBOL}}.onnx")

In [ ]:
# Cell 10: Save to Google Drive (backup)

import shutil

print("[5/7] Saving to Google Drive...")

try:
    from google.colab import drive
    drive.mount("/content/drive")

    save_dir = "/content/drive/MyDrive/PutStrike"
    os.makedirs(save_dir, exist_ok=True)

    for f in ["itransformer.onnx", "model_config.json", "training_results.png"]:
        if os.path.exists(f):
            shutil.copy(f, os.path.join(save_dir, f))

    # Also save PyTorch checkpoint for fine-tuning
    torch.save({
        "model_state_dict": model.state_dict(),
        "feature_names": feature_names,
        "norm_stats": norm_stats,
        "history": history,
    }, os.path.join(save_dir, "checkpoint.pt"))

    print(f"  Saved to: {save_dir}/")
    print(f"  Files: itransformer.onnx, model_config.json, checkpoint.pt, training_results.png")
except ImportError:
    print("  [SKIP] Not running in Google Colab")
except Exception as e:
    print(f"  [WARNING] Drive save failed: {e}")

In [ ]:
# Cell 11: Final Summary & Website Integration Instructions

print("=" * 60)
print("  PutStrike iTransformer — Training Complete!")
print("=" * 60)

print(f"""
  Model: iTransformer v3.0 (Universal)
  Stocks trained: {len(SCREENER_SYMBOLS)}
  Features: {len(feature_names)}
  Parameters: {sum(p.numel() for p in model.parameters()):,}
  ONNX size: {os.path.getsize('itransformer.onnx') / 1024 / 1024:.1f} MB

  Test Directional Accuracy:
    7-day:  {test_metrics['dir_acc_7d']:.1f}%
    14-day: {test_metrics['dir_acc_14d']:.1f}%
    30-day: {test_metrics['dir_acc_30d']:.1f}%
    60-day: {test_metrics['dir_acc_60d']:.1f}%

  Next Steps:
  1. Set HF_REPO_ID in your website's .env:
     NEXT_PUBLIC_HF_REPO_ID={HF_REPO_ID}
  2. The website will auto-download the ONNX model from HuggingFace
  3. Predictions run via onnxruntime-web (WASM, no GPU needed)
  4. Re-run this notebook periodically to retrain with fresh data
""")

print("[7/7] Done!")